# D5: engine loop and memory budgeting

D4 chose work and reserved KV blocks. We still supplied token histories and executed each forward ourselves. D5 connects those pieces into an **inference engine**: a loop that schedules requests, runs our model, selects outputs and releases finished requests.

We will first run the tests, then check real TinyLlama against independent C2 and Hugging Face references. Worked traces cover new arrivals, recomputation, failure/retry, seeded replay and cancellation. Finally we count exact KV bytes and compare FP16 scores with FP32 scores from the same rounded weights.

Run top to bottom on a **fresh Colab T4**. Local checks are static only. This lesson adds no HTTP server and makes no throughput claim. Save the final JSON and this notebook with outputs.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd /content/tinyserve
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)

## 1. Start with the executable specification

The next cell runs the model, generation, batching, cache, scheduler and engine tests together. A failed test stops this lesson. The existing two-GPU test can skip on one T4; the CUDA engine tests should run.

D5's tests check output counts, final-row routing, chunked prompts, mixed arrivals, EOS, atomic admission, cancellation, memory rounding and generator state. Real small Llama models test HF/C2 greedy parity, later-layer failure/retry and FP16-versus-FP32 numerical agreement. Scripted models make lifecycle edge cases deterministic. Their simulated cache commits are policy tests, not evidence that real attention works.

We disable TF32 to keep the FP32 reference setting consistent with earlier parity lessons.

In [ ]:
import gc, json
from dataclasses import asdict
import torch
from tinyserve.basics import load_model, encode
from tinyserve.engine.engine import InferenceEngine
from tinyserve.engine.memory import plan_kv_memory
from tinyserve.generation import generate
from tinyserve.models.attention import AttentionMetadata
from tinyserve.models.llama import LlamaForCausalLM
from tinyserve.envinfo import collect_env, save_result

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
print(collect_env())
test_files = ["tests/test_b8_model.py", "tests/test_c1_sampling.py",
              "tests/test_c2_generation.py", "tests/test_c3_cache.py",
              "tests/test_d1_batching.py", "tests/test_d2_paged_cache.py",
              "tests/test_d3_block_manager.py", "tests/test_d4_scheduler.py",
              "tests/test_d5_engine.py"]
test_run = subprocess.run([sys.executable, "-m", "pytest", *test_files, "-v", "-ra"],
                          capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()

## 2. The off-by-one that matters

A logits row predicts the token **after** its input position. For a three-token prompt and three requested outputs:

| Forward | Positions processed | Output selected |
|---|---|---|
| Prefill | 0, 1, 2 | New token at position 3 |
| Decode | 3 | New token at position 4 |
| Decode | 4 | New token at position 5 |

Only five tokens are processed; six token IDs are returned. In general, `P` prompt tokens and `G>0` outputs need `P+G-1` model rows before recomputation. D4 therefore receives `decode_steps=G-1`. The final output needs no KV slot because it will not be processed again.

**Partial prefill** means only a chunk of the prompt has been processed. It must not sample yet. **Recomputation** rebuilds K/V for tokens already processed; it must neither sample nor consume random draws. The engine retains each request's prompt and previously selected outputs for that replay.

Read `tinyserve/engine/engine.py`: `step()` packs a plan, runs the model once, routes each eligible last row, stages outputs, acknowledges the batch and releases early-EOS work. Empty step updates do not mean the engine is idle; check `has_pending_requests`.

## 3. Load FP32 references and make state visible

The next cell loads TinyLlama twice: HF and our strictly copied model. Both are FP32. This deliberately spends memory on independent correctness references, not on maximum serving capacity.

A snapshot distinguishes logical progress, resident cache length and output history. A sampled output can exist without a resident KV entry. A preempted request keeps logical progress and history but loses resident blocks.

Our comparison helper checks **every planned logits row** against separate no-cache ours/HF forwards, including replay rows. The tolerance is `abs(error) <= 1e-4 + 1e-5 * abs(reference)`, consistent with earlier real TinyLlama backend differences. We separately require equal argmax choices, and save minimum/maximum row differences rather than one selected number.

In [ ]:
tokenizer, hf_model = load_model(device=DEVICE, dtype=torch.float32)
hf_model.set_attn_implementation("eager")
our_model = LlamaForCausalLM.from_huggingface(hf_model)
prompt_texts = ["The capital of France is", "The capital of Germany is"]
encoded_prompts = [encode(tokenizer, text, DEVICE) for text in prompt_texts]
assert all(tokens.numel() >= 3 for tokens in encoded_prompts)

def value_range(values):
    assert values
    return {"min": min(values), "max": max(values)}

def snapshot(engine, owners):
    pool = engine.cache
    used = [block for table in pool.block_tables.values() for block in table]
    assert len(used) == len(set(used))
    assert set(used).isdisjoint(pool.free_blocks)
    assert set(used).union(pool.free_blocks) == set(range(pool.num_blocks))
    assert pool.ref_counts == tuple(int(block in used) for block in range(pool.num_blocks))
    return {"outputs": {str(owner): asdict(engine.get_output(owner)) for owner in owners},
            "logical_progress": {str(owner): state.processed_tokens
                                 for owner, state in engine.scheduler.requests.items()},
            "resident_lengths": dict(pool.lengths),
            "tables": {str(owner): list(table) for owner, table in pool.block_tables.items()},
            "free_blocks": list(pool.free_blocks)}

@torch.no_grad()
def compare_rows(plan, histories, actual):
    rows, offset = [], 0
    for item in plan.work:
        history = histories[item.request_id][:item.end]
        positions = torch.arange(item.end, device=history.device, dtype=history.dtype)
        no_cache = our_model(history, positions, AttentionMetadata(positions))[item.start:item.end]
        hf_logits = hf_model(history[None], use_cache=False).logits[0, item.start:item.end].float()
        count = item.end - item.start
        scheduled = actual[offset:offset + count]
        offset += count
        torch.testing.assert_close(scheduled, no_cache, rtol=1e-5, atol=1e-4)
        torch.testing.assert_close(scheduled, hf_logits, rtol=1e-5, atol=1e-4)
        assert torch.equal(scheduled.argmax(-1), no_cache.argmax(-1))
        assert torch.equal(scheduled.argmax(-1), hf_logits.argmax(-1))
        rows.append({"request_id": item.request_id, "phase": item.phase,
                     "start": item.start, "end": item.end, "all_row_argmax_equal": True,
                     "ours_row_max_abs_diff": (scheduled - no_cache).abs().amax(-1).tolist(),
                     "hf_row_max_abs_diff": (scheduled - hf_logits).abs().amax(-1).tolist()})
    assert offset == plan.num_tokens
    return rows

## 4. Execute readable timelines, including failure/retry

A **batch plan** contains token ranges with an exclusive end: `[start,end)`. The helper below lets us inspect the plan before handing execution back to `engine.step()`. D4 returns the same pending plan; we never run a second paged forward for comparison.

Instead, a temporary model hook captures the engine's actual logits. After removing that hook, independent no-cache comparisons cannot advance the engine cache. We compare final generated IDs with C2 and HF greedy generation too.

In the failure case, step 1 raises before the second decoder layer. Committed lengths, emitted histories and old prefix entries must stay unchanged. The hook is removed and the same plan is retried. We do not acknowledge a failed forward.

For these finite traces only, EOS stopping is temporarily disabled on both model configurations and restored in `finally`. This guarantees equal four-output workloads; a separate cell tests real early-EOS behavior.

In [ ]:
def run_case(name, prompts, blocks, batch_tokens, late_arrival=False, inject_failure=False):
    previous_eos = our_model.config.eos_token_id, hf_model.config.eos_token_id
    our_model.config.eos_token_id = hf_model.config.eos_token_id = None
    try:
        unit = plan_kv_memory(our_model, 1024**2, 2).bytes_per_block
        engine = InferenceEngine(our_model, blocks * unit, block_size=2,
                                 max_batch_tokens=batch_tokens)
        expected = [tuple(generate(our_model, prompt, 4).tolist()) for prompt in prompts]
        with torch.no_grad():
            expected_hf = [tuple(hf_model.generate(prompt[None], max_new_tokens=4,
                           do_sample=False, eos_token_id=None,
                           pad_token_id=tokenizer.eos_token_id)[0].tolist()) for prompt in prompts]
        assert expected == expected_hf
        owners = [0]
        engine.add_request(0, prompts[0], max_new_tokens=4)
        if not late_arrival:
            engine.add_request(1, prompts[1], max_new_tokens=4)
            owners.append(1)
        steps, failure = [], None
        for tick in range(100):
            if late_arrival and tick == 1:
                assert not engine.get_output(0).finished
                engine.add_request(1, prompts[1], max_new_tokens=4)
                owners.append(1)
            if not engine.has_pending_requests:
                break
            plan = engine.scheduler.schedule()
            before = snapshot(engine, owners)
            histories = {owner: prompts[owner].new_tensor(engine.get_output(owner).token_ids)
                         for owner in owners}
            if inject_failure and tick == 1:
                pool = engine.cache
                live_slots = torch.cat([pool.slot_mapping(
                    torch.arange(length, device=pool.key.device, dtype=torch.int64),
                    torch.full((length,), owner, device=pool.key.device, dtype=torch.int64))
                    for owner, length in pool.lengths.items() if length])
                old_prefix = [storage.flatten(1, 2)[:, live_slots].clone()
                              for storage in (pool.key, pool.value)]
                def fail_forward(module, args):
                    raise RuntimeError("injected D5 later-layer failure")
                hook = our_model.model.layers[1].register_forward_pre_hook(fail_forward)
                try:
                    try:
                        engine.step()
                    except RuntimeError as error:
                        if str(error) != "injected D5 later-layer failure":
                            raise
                        failure = {"error": str(error), "before": before,
                                   "after_failure": snapshot(engine, owners)}
                    else:
                        raise AssertionError("injected forward did not fail")
                finally:
                    hook.remove()
                assert failure["before"] == failure["after_failure"]
                assert engine.scheduler.schedule() is plan
                for storage, saved in zip((pool.key, pool.value), old_prefix):
                    torch.testing.assert_close(storage.flatten(1, 2)[:, live_slots], saved,
                                               rtol=0, atol=0)
            captured = []
            hook = our_model.register_forward_hook(
                lambda module, args, output: captured.append(output.detach().clone()))
            try:
                updates = engine.step()
            finally:
                hook.remove()
            assert len(captured) == 1
            comparisons = compare_rows(plan, histories, captured[0])
            replay_owners = {item.request_id for item in plan.work if item.phase == "recompute"}
            assert replay_owners.isdisjoint(output.request_id for output in updates)
            after = snapshot(engine, owners)
            steps.append({"tick": tick, "plan": asdict(plan), "num_tokens": plan.num_tokens,
                          "before": before, "after": after,
                          "updates": [asdict(output) for output in updates],
                          "comparisons": comparisons})
            print(name, tick, asdict(plan), [asdict(output) for output in updates])
        else:
            raise AssertionError("finite engine work did not terminate")
        outputs = [engine.get_output(owner) for owner in range(2)]
        assert all(output.finish_reason == "length" for output in outputs)
        assert [output.token_ids for output in outputs] == expected == expected_hf
        assert engine.cache.num_free_blocks == blocks and not engine.cache.block_tables
        replayed = sum(item["end"] - item["start"] for step in steps
                       for item in step["plan"]["work"] if item["phase"] == "recompute")
        logical = sum(prompt.numel() + 3 for prompt in prompts)
        assert sum(step["num_tokens"] for step in steps) == logical + replayed
        if name == "preemption":
            assert replayed == 2 and engine.scheduler.requests[1].preemptions == 1
        if inject_failure:
            assert failure is not None
        differences = [difference for step in steps for row in step["comparisons"]
                       for difference in row["hf_row_max_abs_diff"]]
        return {"name": name, "pool_budget": asdict(engine.memory_budget),
                "allocated_bytes": engine.cache.allocated_bytes, "steps": steps,
                "logical_tokens": logical, "recomputed_tokens": replayed,
                "scheduled_tokens": sum(step["num_tokens"] for step in steps),
                "hf_row_max_abs_diff_range": value_range(differences),
                "outputs": [asdict(output) for output in outputs],
                "failure_retry": failure, "final": snapshot(engine, owners)}
    finally:
        our_model.config.eos_token_id, hf_model.config.eos_token_id = previous_eos

## 5. New arrivals, small-pool preemption and retry

The first case admits request 1 after the first forward, while request 0 is still active. A four-row budget chunks the prompts; later plans mix prefill with one-token decode.

The second case uses two-token prompts, four outputs each, three blocks and two slots per block. Each request needs five processed positions and fits alone. Together they force the D4 newest-victim preemption timeline:

| Step | Expected action |
|---|---|
| 0 | Request 0 prefills two tokens; request 1 prefills one |
| 1 | Request 0 decodes; request 1 completes its prompt |
| 2 | Request 0 decodes; request 1 waits for blocks |
| 3 | Preempt request 1; request 0 produces its fourth output and finishes |
| 4 | Request 1 replays two prompt tokens; no new output |
| 5-7 | Request 1 resumes decode and finishes |

Ten logical rows plus two replay rows produce eight outputs total. The third case injects a real forward failure and retries. Every case must release all blocks at completion.

In [ ]:
continuous_blocks = sum((prompt.numel() + 4) // 2 for prompt in encoded_prompts)
continuous_result = run_case("continuous_arrival", encoded_prompts, continuous_blocks,
                             batch_tokens=4, late_arrival=True)
short_prompts = [prompt[:2].clone() for prompt in encoded_prompts]
preemption_result = run_case("preemption", short_prompts, blocks=3, batch_tokens=3)
retry_result = run_case("failure_retry", short_prompts, blocks=4, batch_tokens=3,
                       inject_failure=True)

## 6. Seeded sampling survives recomputation

Each request receives its own CUDA `torch.Generator`. This makes its random stream independent of another request's scheduling order. Seeds reproduce behavior only on the same device and software; they do not promise cross-device token identity.

The next cell runs the constrained preemption workload twice with identical seeds, temperature, top-k and top-p. Histories and final generator states must match, each request must emit exactly four tokens, and request 1 must be preempted once. The deterministic tests also compare generator states with a no-cache sampler to catch extra draws during replay.

In [ ]:
def seeded_trial():
    previous_eos = our_model.config.eos_token_id
    our_model.config.eos_token_id = None
    try:
        unit = plan_kv_memory(our_model, 1024**2, 2).bytes_per_block
        engine = InferenceEngine(our_model, 3 * unit, block_size=2, max_batch_tokens=3)
        generators = [torch.Generator(device=DEVICE).manual_seed(71 + owner) for owner in range(2)]
        for owner, prompt in enumerate(short_prompts):
            engine.add_request(owner, prompt, 4, temperature=0.8, top_k=20, top_p=0.9,
                               generator=generators[owner])
        for tick in range(30):
            if not engine.has_pending_requests:
                break
            engine.step()
        else:
            raise AssertionError("seeded workload did not finish")
        outputs = [asdict(engine.get_output(owner)) for owner in range(2)]
        assert all(len(output["generated_tokens"]) == 4 for output in outputs)
        assert engine.scheduler.requests[1].preemptions == 1
        assert engine.cache.num_free_blocks == 3
        return outputs, [generator.get_state().clone() for generator in generators]
    finally:
        our_model.config.eos_token_id = previous_eos

seeded_first, first_states = seeded_trial()
seeded_second, second_states = seeded_trial()
assert seeded_first == seeded_second
assert all(torch.equal(first, second) for first, second in zip(first_states, second_states))
seeded_result = {"seeds": [71, 72], "temperature": 0.8, "top_k": 20, "top_p": 0.9,
                 "first": seeded_first, "second": seeded_second, "rng_states_equal": True,
                 "newer_request_preemptions_each_run": 1}
print(seeded_result)

## 7. EOS, zero output budgets, invalid requests and cancellation

EOS means **end of sequence**. To exercise it without relying on a model deciding to stop naturally, we first calculate the next greedy token, temporarily use that ID as the model's EOS configuration, then request four outputs. The engine must include that token, stop after one output and release the remaining work. The original configuration is restored.

A zero-output request returns its prompt without a forward or reservation. Invalid context and pool-size requests must raise before admission, leaving ownership unchanged. Cancellation is tested before admission to a batch and after one output, preserving already emitted tokens. These are explicit errors and lifecycle checks, not fallback success.

In [ ]:
def lifecycle_checks():
    unit = plan_kv_memory(our_model, 1024**2, 2).bytes_per_block
    prompt = short_prompts[0]
    previous_eos = our_model.config.eos_token_id
    with torch.no_grad():
        positions = torch.arange(prompt.numel(), device=prompt.device, dtype=prompt.dtype)
        forced_eos = our_model(prompt, positions, AttentionMetadata(positions))[-1].argmax().item()
    try:
        our_model.config.eos_token_id = forced_eos
        eos_engine = InferenceEngine(our_model, 3 * unit, block_size=2, max_batch_tokens=3)
        eos_engine.add_request(0, prompt, 4)
        eos_output, = eos_engine.step()
        assert eos_output.generated_tokens == (forced_eos,) and eos_output.finish_reason == "eos"
        assert not eos_engine.has_pending_requests and eos_engine.cache.num_free_blocks == 3
        our_model.config.eos_token_id = None
        engine = InferenceEngine(our_model, 3 * unit, block_size=2, max_batch_tokens=3)
        zero = engine.add_request(20, prompt, 0)
        assert zero.token_ids == tuple(prompt.tolist()) and zero.finished
        assert not engine.scheduler.requests and engine.cache.num_free_blocks == 3
        errors = {}
        for name, budget in [("context", our_model.config.max_position_embeddings), ("pool", 6)]:
            before = snapshot(engine, [20])
            try:
                engine.add_request(1, prompt, budget)
            except ValueError as error:
                errors[name] = str(error)
            else:
                raise AssertionError("invalid request was admitted")
            assert snapshot(engine, [20]) == before
        engine.add_request(0, prompt, 4)
        engine.add_request(1, short_prompts[1], 4)
        waiting = engine.cancel(1)
        assert waiting.generated_tokens == () and waiting.finish_reason == "cancelled"
        engine.step()
        running = engine.cancel(0)
        assert len(running.generated_tokens) == 1 and running.finish_reason == "cancelled"
        assert not engine.has_pending_requests and engine.cache.num_free_blocks == 3
        return {"forced_eos": forced_eos, "eos_output": asdict(eos_output),
                "zero_output": asdict(zero), "admission_errors": errors,
                "waiting_cancel": asdict(waiting), "running_cancel": asdict(running),
                "final_free_blocks": engine.cache.num_free_blocks}
    finally:
        our_model.config.eos_token_id = previous_eos

lifecycle_result = lifecycle_checks()
print(lifecycle_result)

## 8. Budget the KV pool, not the entire process

One token stores **keys and values** at every layer: `2 * layers * KV_heads * head_dim * element_bytes`. TinyLlama uses 22 layers, **4 KV heads** and head dimension 64. Do not multiply by its 32 query heads.

| Dtype | Bytes per token | KiB per token | Bytes per 16-slot block |
|---|---:|---:|---:|
| FP32 | 45056 | 44 | 720896 |
| FP16 | 22528 | 22 | 360448 |

A 1 MiB FP16 KV budget fits two blocks: `1048576 // 360448 = 2`. They allocate 720896 bytes and leave 327680 budget bytes unused. This remainder is whole-block rounding, not unused slots in live requests. Each live last block can also waste up to 15 slots.

`plan_kv_memory()` only calculates. It does not allocate. The engine checks its actual K/V tensor bytes against the plan. Releasing requests makes blocks reusable without shrinking those tensors.

For CUDA, `InferenceEngine.from_available_memory(model, reserve_bytes=...)` queries driver free memory **after weights are loaded**, subtracts explicit workspace headroom and rounds down. The next cell shows the calculation without allocating all available memory. A 1 GiB reserve is an example, not an OOM guarantee: activations, full vocabulary logits, gather temporaries, allocator overhead and other users still matter. Driver-free memory excludes PyTorch-reserved memory, and the snapshot can immediately become stale.

In [ ]:
fp32_plan = plan_kv_memory(our_model, 2 * 1024**2, block_size=16)
assert fp32_plan.bytes_per_token == 45056 and fp32_plan.bytes_per_block == 720896
free_bytes, total_bytes = torch.cuda.mem_get_info(our_model.get_input_embeddings().weight.device)
workspace_reserve = 1024**3
if free_bytes <= workspace_reserve:
    raise RuntimeError("Free memory cannot cover the example workspace reserve; inspect live allocations.")
available_plan = plan_kv_memory(our_model, free_bytes - workspace_reserve, block_size=16)
assert available_plan.allocated_bytes <= free_bytes - workspace_reserve
assert 0 <= available_plan.unused_bytes < available_plan.bytes_per_block
memory_result = {"fp32": asdict(fp32_plan), "fp32_allocated_bytes": fp32_plan.allocated_bytes,
                 "driver_free_bytes": free_bytes, "driver_total_bytes": total_bytes,
                 "example_workspace_reserve_bytes": workspace_reserve,
                 "available_plan_without_allocation": asdict(available_plan)}
print(memory_result)

## 9. FP16 storage and FP32 scores from identical rounded weights

We have finished the HF comparisons. Delete that reference, cast our model to FP16 and build a fresh FP32 model from its **rounded FP16 weights**. This separates arithmetic error from weight-rounding differences and leaves the old engines unused; an engine cannot survive a model dtype change.

The next helper checks every scheduled FP16 logits row against that FP32 model using `rtol=atol=1e-2`, the C3 FP16 tolerance. It checks that the engine selects its own FP16 argmax. It does **not** demand bit-equal FP16/FP32 scores or token identity near a numerical tie. Minimum/maximum row differences and final release are recorded.

The actual small pool must match its planned tensor bytes. The separate 1 MiB plan must reproduce the two-block, 327680-byte remainder calculation.

In [ ]:
del hf_model
gc.collect()
torch.cuda.empty_cache()
our_model.half()
rounded_fp32 = LlamaForCausalLM(our_model.config, device=DEVICE, dtype=torch.float32).eval()
rounded_fp32.load_state_dict(our_model.state_dict(), strict=True)

def fp16_check():
    unit = plan_kv_memory(our_model, 1024**2, 2).bytes_per_block
    engine = InferenceEngine(our_model, 3 * unit, block_size=2, max_batch_tokens=3)
    engine.add_request(0, short_prompts[0], 4)
    differences, checked_rows = [], 0
    for tick in range(30):
        if not engine.has_pending_requests:
            break
        plan = engine.scheduler.schedule()
        history = short_prompts[0].new_tensor(engine.get_output(0).token_ids)
        captured = []
        hook = our_model.register_forward_hook(
            lambda module, args, output: captured.append(output.detach().clone()))
        try:
            updates = engine.step()
        finally:
            hook.remove()
        offset = 0
        for item in plan.work:
            positions = torch.arange(item.end, device=history.device, dtype=history.dtype)
            with torch.no_grad():
                reference = rounded_fp32(history[:item.end], positions,
                                         AttentionMetadata(positions))[item.start:item.end]
            count = item.end - item.start
            actual = captured[0][offset:offset + count]
            offset += count
            torch.testing.assert_close(actual, reference, rtol=1e-2, atol=1e-2)
            differences.extend((actual - reference).abs().amax(-1).tolist())
            checked_rows += count
            if updates:
                assert updates[0].generated_tokens[-1] == actual[-1].argmax().item()
    else:
        raise AssertionError("FP16 work did not finish")
    assert engine.cache.allocated_bytes == engine.memory_budget.allocated_bytes
    assert engine.cache.num_free_blocks == 3
    plan_1mib = plan_kv_memory(our_model, 1024**2, 16)
    assert plan_1mib.bytes_per_token == 22528 and plan_1mib.num_blocks == 2
    assert plan_1mib.allocated_bytes == 720896 and plan_1mib.unused_bytes == 327680
    return {"rtol": 1e-2, "atol": 1e-2, "checked_rows": checked_rows,
            "row_max_abs_diff_range": value_range(differences),
            "pool_budget": asdict(engine.memory_budget),
            "actual_allocated_bytes": engine.cache.allocated_bytes,
            "one_mib_plan": asdict(plan_1mib), "one_mib_unused_bytes": plan_1mib.unused_bytes,
            "output": asdict(engine.get_output(0)), "final_free_blocks": 3}

fp16_result = fp16_check()
print(fp16_result)
del rounded_fp32
gc.collect()
torch.cuda.empty_cache()

## 10. Save complete evidence and explain it

The final cell saves test stdout/stderr, all three FP32 traces, failure snapshots, seeded replay, lifecycle errors and both memory calculations. `save_result()` adds the original timestamp, GPU, library versions and source SHA. No local static check substitutes for this runtime evidence.

Keep both `results/D5_engine.json` and this notebook with outputs. Share the complete printed JSON to record validation in the repository. A failing assertion means D5 remains pending; do not mark it complete from a partial run.

### Three interview questions
1. Why do `G` generated tokens need only `G-1` decode forwards? The final prompt row predicts the first output; the last output is returned without processing it.
2. Why must replay avoid sampling? It rebuilds K/V for already chosen tokens. Sampling again duplicates outputs and consumes RNG state twice.
3. Why can a correctly sized KV pool still OOM? Weights and variable workspace are separate; driver-free memory is a changing snapshot.

Guide: `docs/guide/20-engine-loop.md`. Next: E1, streaming OpenAI-compatible serving from Colab. Prefix sharing, copy-on-write, kernels, CUDA graphs and serving benchmarks remain deferred.

In [ ]:
assert test_run.returncode == 0
assert preemption_result["recomputed_tokens"] == 2
assert retry_result["failure_retry"] is not None
d5_result = {"test_run": {"returncode": test_run.returncode, "stdout": test_run.stdout,
                          "stderr": test_run.stderr},
             "fp32_backend": "pytorch_paged_gather", "fp32_rtol": 1e-5, "fp32_atol": 1e-4,
             "scenarios": [continuous_result, preemption_result, retry_result],
             "seeded_replay": seeded_result, "lifecycle": lifecycle_result,
             "memory": memory_result, "fp16": fp16_result}
result_path = save_result("D5_engine", d5_result)
print(result_path.read_text())